# Exercise 07: Unsupervised Learning – Clustering Countries with k-Means, Hierarchical Clustering and PCA

**Dataset:** socio-economic and health indicators of 167 countries, compiled for the NGO *HELP International*
(about 2017, based on World Bank data). Source:
[Kaggle – Unsupervised Learning on Country Data](https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data).

| variable | meaning |
|---|---|
| `child_mort` | deaths of children under 5 per 1'000 live births |
| `exports`, `imports` | exports / imports of goods and services per capita (% of GDP per capita) |
| `health` | total health spending per capita (% of GDP per capita) |
| `income` | net income per person (USD, purchasing power adjusted) |
| `inflation` | annual growth rate of GDP (inflation, %) |
| `life_expec` | life expectancy at birth (years) |
| `total_fer` | number of children per woman (fertility) |
| `gdpp` | GDP per capita (USD) |

**Business question:** HELP International has raised 10 million USD and wants to decide which countries need the aid
most urgently. There is **no target variable** – we look for groups of similar countries and for a compact description
of the nine indicators.

**Slides:** Part 05 «Unsupervised learning» – k-means, elbow method, hierarchical clustering (dendrogram), PCA.

**After this exercise you can ...**
- perform k-means iterations by hand and compute the within-cluster sum of squares,
- apply k-means (incl. scaling, elbow plot, several random starts) and interpret clusters,
- create and cut a dendrogram,
- use PCA to summarise many correlated features and to visualise clusters.

## Concept

**k-means.** (1) choose $k$; (2) place $k$ centres; (3) assign every point to its nearest centre; (4) move each centre to
the mean of its points; repeat (3)–(4) until nothing changes. k-means minimises the within-cluster sum of squares

$$\text{WCSS} = \sum_{c=1}^{k}\;\sum_{x_i \in C_c} \lVert x_i - \mu_c \rVert^2 \qquad(\texttt{inertia\_} \text{ in scikit-learn})$$

The result depends on the random start → use several starts (`n_init`). **Elbow method:** plot WCSS against $k$; more
clusters always reduce it – choose $k$ where the curve bends. Scale the features first and check that the clusters make
sense for the business question.

**Hierarchical (agglomerative) clustering.** Start with every observation as its own cluster; repeatedly merge the two
closest clusters until one cluster is left. The **dendrogram** shows the merge history (height = distance at which the
clusters were merged); cutting it at a height gives the clusters. *Ward linkage* merges the two clusters that increase the
WCSS the least.

**PCA (principal component analysis).** Rotates the coordinate system so that the first new axis (PC1) captures as much
of the variance as possible, PC2 the largest part of the remaining variance (orthogonal to PC1), etc. The **loadings**
show how each PC is built from the original (standardised) features; the **explained variance ratio** shows how much of
the total variance each PC captures.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-means by hand

Six points in two dimensions: A (1, 1), B (2, 1), C (4, 3), D (5, 4), E (1, 2), F (5, 5).
We use $k = 2$ with the initial centres $\mu_1 = A = (1, 1)$ and $\mu_2 = C = (4, 3)$.

a) Assign every point to its nearest centre (Euclidean distance).

b) Compute the new centres.

c) Repeat the assignment step. Has the algorithm converged?

d) Compute the WCSS of the final solution.

e) Verify your results with `KMeans(n_clusters=2, init=np.array([[1, 1], [4, 3]]), n_init=1)`.

In [ ]:
points = pd.DataFrame({'x1': [1, 2, 4, 5, 1, 5], 'x2': [1, 1, 3, 4, 2, 5]}, index=list('ABCDEF'))
centres = np.array([[1.0, 1.0], [4.0, 3.0]])

for it in range(1, 4):
    dist = np.sqrt(((points.values[:, None, :] - centres[None, :, :])**2).sum(axis=2))
    labels = dist.argmin(axis=1)
    print(f'Iteration {it}: distances to centres =')
    print(pd.DataFrame(dist.round(2), index=points.index, columns=['to mu1', 'to mu2']).T)
    print('  assignment:', {p: int(l) + 1 for p, l in zip(points.index, labels)})
    new_centres = np.array([points.values[labels == c].mean(axis=0) for c in range(2)])
    print('  new centres:', new_centres.round(3).tolist(), '\n')
    if np.allclose(new_centres, centres):
        print('Converged: the centres do not change any more.')
        break
    centres = new_centres

wcss = sum(((points.values[labels == c] - centres[c])**2).sum() for c in range(2))
print('WCSS =', round(wcss, 3))

# Check with scikit-learn
km_check = KMeans(n_clusters=2, init=np.array([[1, 1], [4, 3]]), n_init=1).fit(points)
print('scikit-learn: centres =', km_check.cluster_centers_.round(3).tolist(), ' inertia =', round(km_check.inertia_, 3))

**Solution:**

a) Distances to $\mu_1=(1,1)$ / $\mu_2=(4,3)$: A 0 / 3.61, B 1 / 2.83, C 3.61 / 0, D 5 / 1.41, E 1 / 3.16, F 5.66 / 2.24
→ cluster 1 = {A, B, E}, cluster 2 = {C, D, F}.

b) $\mu_1 = \big(\tfrac{1+2+1}{3}, \tfrac{1+1+2}{3}\big) = (1.33, 1.33)$,
$\mu_2 = \big(\tfrac{4+5+5}{3}, \tfrac{3+4+5}{3}\big) = (4.67, 4)$.

c) With the new centres all points keep their cluster → the centres do not change any more; the algorithm has converged
after the second assignment.

d) Cluster 1: A $(0.11+0.11)$ + B $(0.44+0.11)$ + E $(0.11+0.44)$ = 1.33;
cluster 2: C $(0.44+1)$ + D $(0.11+0)$ + F $(0.11+1)$ = 2.67 → **WCSS = 4.0**.

## Exercise 2: Import and explore the country data

a) Import the data, set `country` as index and show the descriptive statistics. Which countries have the highest and the
lowest child mortality?

b) Plot a correlation heatmap of the nine indicators. Which indicators are strongly correlated?

c) Why must the features be standardised before clustering?

In [ ]:
# a) Import the data
countries = pd.read_csv('./Data/country_data.csv', sep=',', index_col='country')
print(countries.shape)
print(countries.describe().round(1).T)
print('\nHighest child mortality:', countries['child_mort'].nlargest(5).to_dict())
print('Lowest child mortality: ', countries['child_mort'].nsmallest(5).to_dict())

# b) Correlation heatmap
plt.figure(figsize=(8, 6.5))
sns.heatmap(countries.corr(), annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, square=True)
plt.title('Correlation matrix of the country indicators')
plt.show()

**Solution:**

a) Child mortality ranges from about 2.6 (Iceland) to over 200 (Haiti) deaths per 1'000 births; the countries with the
highest values are in sub-Saharan Africa and Haiti, the lowest in Northern and Western Europe (incl. Luxembourg and
Singapore).

b) Strong correlations: `income` and `gdpp` (≈ 0.9); `child_mort` and `total_fer` (≈ 0.85); `child_mort` and
`life_expec` (≈ −0.89); `exports` and `imports` (≈ 0.74). Many indicators measure the same underlying dimension
("level of development") – a good situation for PCA.

c) The units differ enormously (income and GDP in USD with standard deviations in the tens of thousands, fertility in
children per woman with a standard deviation of about 1.5). Without scaling, k-means (Euclidean distance) would cluster
almost only by `income` and `gdpp`.

## Exercise 3: k-means – choosing k

a) Standardise the nine indicators with `StandardScaler`.

b) Fit `KMeans(n_clusters=k, n_init=10, random_state=42)` for $k = 1, \dots, 10$ and plot the WCSS (`inertia_`)
against $k$ (elbow plot). Additionally compute the silhouette score for $k = 2, \dots, 10$.

c) Which $k$ do you choose? Justify your choice also with regard to the business question.

In [ ]:
# a) Standardise
scaler = StandardScaler()
X = scaler.fit_transform(countries)

# b) Elbow plot and silhouette score
ks = range(1, 11)
wcss, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_) if k > 1 else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ks, wcss, marker='o')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('WCSS (inertia)')
axes[0].set_title('Elbow plot')
axes[1].plot(ks, sil, marker='o', color='darkred')
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette score')
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()
print(pd.DataFrame({'k': ks, 'WCSS': np.round(wcss, 1), 'silhouette': np.round(sil, 3)}).to_string(index=False))

**Solution:**

The elbow is not sharp (typical for real data): the WCSS drops strongly from $k=1$ to $k=3$ and then decreases more
slowly. The silhouette scores for $k = 2$ to 5 are all similar (≈ 0.28–0.30) and decrease for larger $k$. Both criteria
do not give a unique answer. **$k = 3$** is a sensible choice: it is at the bend of the elbow, the clusters can be
interpreted easily ("developing / emerging / developed countries"), and for the NGO's question one group of countries in
need is exactly what is required. $k = 4$ or 5 would also be defensible – the final choice is a domain decision.

## Exercise 4: Interpret the clusters

a) Fit k-means with $k = 3$ (`n_init=10, random_state=42`) and add the cluster labels to the data frame.

b) Compute the mean of each indicator per cluster (in the **original units**) and the number of countries per cluster.
Give each cluster a meaningful name.

c) Plot `gdpp` (logarithmic axis) against `child_mort`, coloured by cluster, and label a few countries (e.g.
Switzerland, United States, India, Nigeria, Haiti, Qatar).

d) In which cluster are Switzerland, India and Nigeria?

In [ ]:
# a) k-means with k = 3
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
countries['cluster'] = km3.labels_

# b) Cluster profiles
profile = countries.groupby('cluster').mean().round(1)
profile['n_countries'] = countries['cluster'].value_counts().sort_index()
print(profile.T)

# c) Scatter plot gdpp vs. child_mort
label_countries = ['Switzerland', 'United States', 'India', 'Nigeria', 'Haiti', 'Qatar',
                   'Brazil', 'China', 'Luxembourg', 'Singapore', 'Central African Republic']
plt.figure(figsize=(9, 5.5))
sns.scatterplot(data=countries, x='gdpp', y='child_mort', hue='cluster', palette='tab10', s=40)
for c in label_countries:
    plt.annotate(c, (countries.loc[c, 'gdpp'], countries.loc[c, 'child_mort']),
                 fontsize=8, xytext=(4, 4), textcoords='offset points')
plt.xscale('log')
plt.xlabel('GDP per capita [USD, log scale]')
plt.ylabel('Child mortality per 1000 births')
plt.title('k-means clusters (k = 3)')
plt.grid(alpha=0.3)
plt.show()

# d) Cluster of selected countries
print(countries.loc[['Switzerland', 'India', 'Nigeria'], 'cluster'])

**Solution:**

b) The three clusters (the cluster numbers 0/1/2 are arbitrary and can differ between runs):

| cluster | typical values | name | countries |
|---|---|---|---|
| high child mortality (≈ 90), low life expectancy (≈ 59 years), ≈ 5 children per woman, GDP per capita ≈ 2'000 USD | **developing countries / in need of aid** | ≈ 47 |
| child mortality ≈ 22, life expectancy ≈ 73 years, GDP per capita ≈ 6'500 USD | **emerging countries** | ≈ 84 |
| child mortality ≈ 5, life expectancy ≈ 80 years, GDP per capita > 40'000 USD | **developed countries** | ≈ 36 |

c)/d) The clusters form bands along the "development axis": the poorer a country, the higher the child mortality.
Switzerland belongs to the developed countries, India to the emerging countries, Nigeria to the countries in need.
The boundaries between the clusters are fluid – countries close to a boundary could just as well belong to the
neighbouring cluster.

## Exercise 5: Stability of k-means

k-means depends on the random initial centres.

a) Fit k-means with $k = 3$ and `n_init=1` for `random_state` = 0, 1, …, 19. Print the WCSS of each run and compare the
cluster assignment with the solution of Exercise 4 using the *adjusted Rand index* (ARI: 1 = identical partition,
independent of the cluster numbers; ≈ 0 = random agreement).

b) What do you conclude? Why is `n_init=10` (the default in the slides) a good idea?

In [ ]:
rows = []
for seed in range(20):
    km = KMeans(n_clusters=3, n_init=1, random_state=seed).fit(X)
    rows.append({'random_state': seed, 'WCSS': round(km.inertia_, 1),
                 'ARI vs. reference': round(adjusted_rand_score(km3.labels_, km.labels_), 3)})
stability = pd.DataFrame(rows)
print(stability.to_string(index=False))
print('\nReference solution (n_init=10): WCSS =', round(km3.inertia_, 1))

**Solution:**

Most runs end in the same solution as the reference (same WCSS, ARI = 1). Some runs get stuck in a worse *local*
minimum (higher WCSS) with a clearly different partition (ARI < 1). k-means only guarantees a local optimum. With
`n_init=10` the algorithm is started 10 times and the solution with the smallest WCSS is kept – this makes the result
much more reliable. Stable clusters (the same solution in many runs) are also an indication that the structure is real.

## Exercise 6: Hierarchical clustering and dendrogram

a) Compute the hierarchical clustering of the standardised data with `linkage(X, method='ward')` and draw the dendrogram
with the country names as labels (`dendrogram(..., labels=countries.index.tolist(), leaf_font_size=6)`, use a wide
figure).

b) Cut the dendrogram into 3 clusters (`fcluster(Z, t=3, criterion='maxclust')`) and compare them with the k-means
clusters using `pd.crosstab`. How similar are the two solutions?

c) At which height would you cut the dendrogram? What does a long vertical line in the dendrogram mean?

In [ ]:
# a) Ward linkage and dendrogram
Z = linkage(X, method='ward')
plt.figure(figsize=(22, 7))
dendrogram(Z, labels=countries.index.tolist(), leaf_font_size=6, color_threshold=Z[-3, 2])
plt.ylabel('Merge distance (Ward)')
plt.title('Dendrogram of 167 countries (Ward linkage, standardised indicators)')
plt.show()

# b) Cut into 3 clusters and compare with k-means
countries['hc_cluster'] = fcluster(Z, t=3, criterion='maxclust')
print(pd.crosstab(countries['cluster'], countries['hc_cluster'],
                  rownames=['k-means'], colnames=['hierarchical']))
print('ARI k-means vs. hierarchical:', round(adjusted_rand_score(countries['cluster'], countries['hc_cluster']), 3))

**Solution:**

b) The two methods agree for most countries: the developed countries and most of the countries in need are assigned to
the same group. The differences lie at the boundaries: hierarchical clustering assigns several countries that k-means
calls "in need" to a group with emerging countries, and a few emerging countries to the developed countries. The
partitions are similar but not identical (ARI clearly above 0, but below 1) – clustering results always depend on the
method.

c) A long vertical line means that the clusters below it are only merged at a large distance – they are clearly
separated. A good cut lies where the vertical lines are long, i.e. where a horizontal cut crosses few lines over a wide
height range. Here, a cut into 3 (or 4) clusters is plausible; the colours in the plot show the 3-cluster cut.

## Exercise 7: PCA – summarising nine indicators

a) Fit a PCA on the standardised data. Plot the explained variance ratio of each component and the cumulative explained
variance. How many components are needed for 80 % of the variance?

b) Show the loadings of PC1 and PC2 (`pca.components_`) as a table. Interpret PC1 and PC2.

c) Plot the countries in the PC1–PC2 plane, coloured by k-means cluster, and label the countries from Exercise 4.

In [ ]:
features = ['child_mort', 'exports', 'health', 'imports', 'income',
            'inflation', 'life_expec', 'total_fer', 'gdpp']

# a) PCA and explained variance
pca = PCA()
scores = pca.fit_transform(X)
evr = pca.explained_variance_ratio_
pcs = [f'PC{i}' for i in range(1, len(evr) + 1)]
print(pd.DataFrame({'explained variance': evr.round(3), 'cumulative': evr.cumsum().round(3)}, index=pcs).T)

plt.figure(figsize=(7, 4))
plt.bar(pcs, evr, color='steelblue', label='per component')
plt.plot(pcs, evr.cumsum(), marker='o', color='darkred', label='cumulative')
plt.axhline(0.8, color='grey', linestyle='--', linewidth=1)
plt.ylabel('Share of explained variance')
plt.legend()
plt.show()

# b) Loadings of PC1 and PC2
loadings = pd.DataFrame(pca.components_[:2].T, index=features, columns=['PC1', 'PC2'])
print(loadings.round(2).sort_values('PC1'))

# c) Countries in the PC1-PC2 plane
plt.figure(figsize=(9, 6))
sns.scatterplot(x=scores[:, 0], y=scores[:, 1], hue=countries['cluster'], palette='tab10', s=40)
for c in label_countries:
    i = countries.index.get_loc(c)
    plt.annotate(c, (scores[i, 0], scores[i, 1]), fontsize=8, xytext=(4, 4), textcoords='offset points')
plt.xlabel(f'PC1 ({evr[0]:.0%} of variance)')
plt.ylabel(f'PC2 ({evr[1]:.0%} of variance)')
plt.title('Countries in the plane of the first two principal components')
plt.grid(alpha=0.3)
plt.show()

**Solution:**

a) PC1 explains about **46 %** of the variance, PC2 about 17 %, PC3 about 13 %. Four components are needed for more than
80 % (≈ 87 %). Nine indicators can therefore be summarised well by a few components.

b) **PC1 = "level of development":** large positive loadings for income, GDP per capita and life expectancy, large
negative loadings for child mortality and fertility. Countries with a high PC1 score are rich and healthy.
**PC2 = "openness to trade":** dominated by exports and imports (both positive), with a negative loading for health
spending and life expectancy. Small, very open economies (e.g. Luxembourg, Singapore) have high PC2 values.

c) The k-means clusters are separated almost exclusively along **PC1**: the clusters correspond to sections of the
development axis. Luxembourg and Singapore stand out at the top (extreme trade openness). PCA makes it possible to look at
a 9-dimensional clustering result in a single 2D plot.

## Exercise 8: Recommendation for HELP International

a) Within the cluster "in need of aid", rank the countries by their PC1 score (lowest = least developed) and show the 10
countries with the lowest scores together with `child_mort`, `life_expec`, `income` and `gdpp`.

b) Write a short recommendation (3–4 sentences) for the NGO. Which limitations of the analysis should be mentioned?

In [ ]:
# a) Countries most in need
countries['PC1'] = scores[:, 0]
need_cluster = countries.groupby('cluster')['child_mort'].mean().idxmax()
top10 = (countries[countries['cluster'] == need_cluster]
         .sort_values('PC1')
         .head(10)[['PC1', 'child_mort', 'life_expec', 'income', 'gdpp']])
print(top10.round(2))

**Solution:**

a) The ten countries with the lowest development score are Nigeria, Haiti, Central African Republic, Chad, Niger, Mali,
Sierra Leone, Congo Dem. Rep., Burkina Faso and Malawi – all in sub-Saharan Africa except Haiti.

Look closely at **Nigeria** in first place: its child mortality and life expectancy are bad, but not the worst, and its
income is higher than that of the others. It ranks first mainly because of its extreme **inflation of 104 %** (by far the
highest value in the data), which loads negatively on PC1. Whether inflation should influence the allocation of
humanitarian aid at all is a question for the NGO, not for the algorithm – a ranking based on PC1 is only as good as the
selected features.

b) *Recommendation:* Based on nine socio-economic and health indicators, about 47 of 167 countries form a group with very
high child mortality, low life expectancy and low income. Within this group, the countries listed above show the lowest
development score and should be considered first for the 10 million USD. *Limitations:* clusters and rankings depend on
the selected indicators (see the Nigeria example), their scaling, outliers and the method ($k$, linkage); the data are a snapshot of one year and do not
contain information on population size, political stability or the presence of other aid organisations; finally, an
unsupervised analysis provides a structured shortlist – the decision itself requires domain expertise.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')